# Serve Qianfan-OCR from Colab (vLLM + cloudflared tunnel)
Runtime: **GPU (T4 is enough)**. Run the cells top to bottom. The last cell prints the values for the worker:
`QIANFAN_URL` (base URL, no path) and `QIANFAN_REVISION`. The tunnel URL changes each session; never commit it.
With an API key set, also export `QIANFAN_API_KEY`. Anyone with the tunnel URL can use your GPU unless you set a key.

In [ ]:
!pip install -q vllm huggingface_hub

In [ ]:
#@title Settings
REVISION = ""  #@param {type:"string"}
USE_API_KEY = True  #@param {type:"boolean"}
PORT = 8000

In [ ]:
# Resolve the revision to pin: the given one, else the current commit of the model repo (so you can record it).
from huggingface_hub import HfApi
MODEL = "baidu/Qianfan-OCR"
revision = REVISION.strip() or HfApi().model_info(MODEL).sha
print("revision:", revision)
api_key = None
if USE_API_KEY:
    from getpass import getpass
    api_key = getpass("API key to require (leave empty for none): ") or None

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, urllib.request, urllib.error, json, re, os
cmd = ["vllm", "serve", MODEL, "--revision", revision, "--trust-remote-code",
       "--hf-overrides", json.dumps({"architectures": ["InternVLChatModel"]}),
       "--dtype", "float16", "--port", str(PORT)]
if api_key:
    cmd += ["--api-key", api_key]
vllm = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}
for _ in range(180):
    try:
        urllib.request.urlopen(urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/models", headers=headers), timeout=2)
        break
    except urllib.error.HTTPError:
        break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("vLLM did not come up; see vllm.log")
print("vLLM is up")

In [ ]:
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stderr=subprocess.PIPE, text=True)
url = None
for line in tunnel.stderr:
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
print("Paste into the worker environment:")
print(f"  QIANFAN_URL={url}")
print(f"  QIANFAN_REVISION={revision}")
if api_key:
    print("  QIANFAN_API_KEY=<the key you entered>")
print("  EXTRACTOR=http")